# RoadSense: Indian traffic-sign training with YOLOv8
Run cells in order in Google Colab. Choose **Runtime → Change runtime type → GPU**.

Dataset: [SDI Indian Traffic Sign](https://universe.roboflow.com/sdi/indian-traffic-sign), published as 6,750 images / 85 classes / CC BY 4.0 (checked 17 September 2026). Download its available version as **YOLOv8** to your computer first. Sign-in/export availability is controlled by Roboflow. This notebook uses your downloaded ZIP, not a private API key.

The exported class count and contents are checked below. Its real-road coverage has not been verified here. Do not assume 85 robust classes just because the page lists 85. No training or improved accuracy is claimed by this notebook's delivery.

Start with all exported classes to avoid silently treating omitted classes as background. If reducing classes later, remap labels consistently and annotate/filter images deliberately. Physical traffic-light states need separate labeled data; `TRAFFIC_SIGNAL` is a warning sign.


In [ ]:
%pip install -q "ultralytics>=8.3,<9" "PyYAML>=6,<7"
import torch, ultralytics
print("Ultralytics:", ultralytics.__version__, "Torch:", torch.__version__)
assert torch.cuda.is_available(), "Choose a GPU runtime before training"
print(torch.cuda.get_device_name(0))

## 1. Save results on Drive
Dataset images stay on the Colab disk for faster reads. Training runs and checkpoints go to Drive. Re-upload the dataset after a runtime reset.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import subprocess, sys

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/RoadSense/runs")
PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT.parent / "colab_environment.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))

## 2. Upload only the downloaded dataset ZIP
This cell refuses to overwrite an earlier extracted dataset. To try a revised export, change the extraction directory or start a fresh runtime.

In [ ]:
import zipfile, io, shutil
uploaded = files.upload()
assert len(uploaded) == 1, "Upload just one dataset ZIP"
zip_name, zip_bytes = next(iter(uploaded.items()))
assert zip_name.lower().endswith(".zip"), "Expected a ZIP export"
EXTRACT = Path("/content/roadsense_indian_dataset")
assert not EXTRACT.exists(), "Dataset directory already exists; use a fresh directory"
EXTRACT.mkdir()
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
    for item in archive.infolist():
        target = (EXTRACT / item.filename).resolve()
        assert target.is_relative_to(EXTRACT.resolve()), "Unsafe archive path"
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, "Symlinks are not supported"
    archive.extractall(EXTRACT)
configs = list(EXTRACT.rglob("data.yaml"))
assert len(configs) == 1, f"Expected one data.yaml, found {len(configs)}"
DATA_ROOT = configs[0].parent
print("Dataset root:", DATA_ROOT)
del uploaded, zip_bytes

## 3. Audit labels, class balance, and exact duplicate leakage
Missing labels are treated as errors. For a genuinely empty image, inspect it and supply an empty `.txt`; do not automatically mark unlabeled signs as background. The checker supports standard Roboflow `train/images`, `valid/images` (or `val/images`), and optional `test/images` folders. It deliberately rebuilds absolute dataset paths for Colab.

In [ ]:
"""Audit a standard Roboflow YOLO detection export without altering its splits.

Usage: python audit_dataset.py /path/to/export
Writes audited_data.yaml and audit_report.json beside the source data.yaml.
Strict about missing labels: verified negatives must have empty .txt files.
"""
import argparse
from collections import Counter, defaultdict
import hashlib
import json
import math
from pathlib import Path
import yaml
from PIL import Image

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def audit(root):
    root = Path(root).resolve()
    config = yaml.safe_load((root / "data.yaml").read_text())
    names = config["names"]
    if isinstance(names, dict):
        names = {int(k): str(v) for k, v in names.items()}
        if sorted(names) != list(range(len(names))):
            raise ValueError("Class IDs must be contiguous from zero")
        names = [names[i] for i in range(len(names))]
    if not isinstance(names, list) or not names or len(set(names)) != len(names):
        raise ValueError("Expected nonempty unique class names")
    if "nc" in config and int(config["nc"]) != len(names):
        raise ValueError("nc does not match names")
    report = {"names": names, "splits": {}, "errors": [], "warnings": []}
    hashes = defaultdict(list)
    data = {"path": str(root), "names": names, "nc": len(names)}
    for split, folders in (("train", ["train"]), ("val", ["valid", "val"]), ("test", ["test"])):
        options = [root / f / "images" for f in folders if (root / f / "images").is_dir()]
        if len(options) != 1:
            if split != "test":
                report["errors"].append(f"Expected exactly one {split} images directory")
            else:
                report["warnings"].append("No test split: create an independent labeled road-scene test set")
            continue
        image_dir = options[0]
        data[split] = str(image_dir)
        paths = sorted(p for p in image_dir.rglob("*") if p.suffix.lower() in EXTENSIONS)
        counts, image_counts = Counter(), Counter()
        empty, small, boxes = 0, 0, 0
        if not paths:
            report["errors"].append(f"Empty {split} image directory")
        for path in paths:
            try:
                with Image.open(path) as im:
                    im = im.convert("RGB")
                    w, h = im.size
                    digest = hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()
                    hashes[digest].append((split, str(path.relative_to(root))))
            except Exception as exc:
                report["errors"].append(f"Unreadable image {path.name}: {exc}")
                continue
            label = image_dir.parent / "labels" / path.relative_to(image_dir).with_suffix(".txt")
            if not label.exists():
                report["errors"].append(f"Missing label: {label.relative_to(root)}")
                continue
            lines = [s for s in label.read_text().splitlines() if s.strip()]
            empty += not lines
            seen = set()
            for line_no, line in enumerate(lines, 1):
                try:
                    values = list(map(float, line.split()))
                    if len(values) != 5 or not all(map(math.isfinite, values)):
                        raise ValueError("expected five finite numbers")
                    cls, x, y, bw, bh = values
                    if not cls.is_integer() or not 0 <= cls < len(names):
                        raise ValueError("invalid class ID")
                    if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
                        raise ValueError("invalid normalized box")
                    if min(x-bw/2, y-bh/2) < -0.001 or max(x+bw/2, y+bh/2) > 1.001:
                        raise ValueError("box extends outside image")
                    counts[int(cls)] += 1
                    seen.add(int(cls))
                    boxes += 1
                    small += min(bw*w, bh*h) * 640/max(w, h) < 16
                except ValueError as exc:
                    report["errors"].append(f"{label.relative_to(root)}:{line_no}: {exc}")
            image_counts.update(seen)
        report["splits"][split] = {
            "images": len(paths), "empty_labels": empty, "boxes": boxes,
            "boxes_with_short_side_under_16px_at_640": small,
            "instances_per_class": {name: counts[i] for i, name in enumerate(names)},
            "images_per_class": {name: image_counts[i] for i, name in enumerate(names)},
        }
        absent = [n for i, n in enumerate(names) if not counts[i]]
        if absent:
            report["errors" if split == "train" else "warnings"].append(f"{split} classes without instances: {absent}")
    cross = [v for v in hashes.values() if len({s for s, _ in v}) > 1]
    report["cross_split_duplicate_groups"] = cross
    report["within_split_duplicate_groups"] = [v for v in hashes.values() if len(v)>1 and len({s for s, _ in v})==1]
    if cross:
        report["errors"].append(f"{len(cross)} exact decoded-image duplicate groups cross splits; repair splits before training")
    report["warnings"].append("Exact hashes do not detect all resized, augmented, near-duplicate images or adjacent video frames. Review source groups manually.")
    (root / "audit_report.json").write_text(json.dumps(report, indent=2))
    if report["errors"]:
        (root / "audited_data.yaml").unlink(missing_ok=True)
    else:
        (root / "audited_data.yaml").write_text(yaml.safe_dump(data, sort_keys=False))
    return report



REPORT = audit(DATA_ROOT)
for split, stats in REPORT["splits"].items():
    print(split, "images:", stats["images"], "boxes:", stats["boxes"], "negative images:", stats["empty_labels"])
    print("Small boxes at 640:", stats["boxes_with_short_side_under_16px_at_640"])
print("Classes:", len(REPORT["names"]), REPORT["names"])
print("Warnings:", *REPORT["warnings"], sep="\n")
print("Errors:", *REPORT["errors"][:30], sep="\n")
shutil.copy(DATA_ROOT / "audit_report.json", PROJECT.parent / "audit_report.json")
assert not REPORT["errors"], "Fix the reported dataset problems before training"
DATA_YAML = DATA_ROOT / "audited_data.yaml"
import pandas as pd
counts = pd.DataFrame({s: d["images_per_class"] for s,d in REPORT["splits"].items()}).fillna(0).astype(int)
display(counts)
counts.to_csv(PROJECT.parent / "class_image_counts.csv")

## 4. Visually inspect images with their boxes
Check small/distant signs, full road scenes, box placement, label correctness, and different locations. If the export is mainly icons or close-up cutouts, treat it as an initial baseline and add labeled real Indian road images before claiming generalization.

The hash check cannot find all augmented copies or adjacent video frames. Keep each original image/video/location group in one split. If repairing a split, work from original images, assign groups roughly 70/15/15, then augment training only. Do not randomly redistribute an already augmented export.


In [ ]:
import random
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
cfg = yaml.safe_load(DATA_YAML.read_text())
for split in ("train", "val"):
    folder = Path(cfg[split])
    candidates = sorted(f for f in folder.rglob("*") if f.suffix.lower() in EXTENSIONS)
    chosen = random.Random(42).sample(candidates, min(12, len(candidates)))
    fig, axes = plt.subplots(3, 4, figsize=(18, 12))
    for ax in axes.flat: ax.axis("off")
    for ax, path in zip(axes.flat, chosen):
        with Image.open(path) as im:
            ax.imshow(im); w,h = im.size
        label = folder.parent / "labels" / path.relative_to(folder).with_suffix(".txt")
        for line in label.read_text().splitlines():
            if not line.strip(): continue
            cls,x,y,bw,bh = map(float,line.split())
            ax.add_patch(Rectangle(((x-bw/2)*w,(y-bh/2)*h),bw*w,bh*h,fill=False,color="lime"))
            ax.text(x*w,(y-bh/2)*h,REPORT["names"][int(cls)],color="black",fontsize=7,backgroundcolor="lime")
        ax.set_title(path.name[:30], fontsize=8)
    fig.suptitle(split + " labeled examples")
    plt.tight_layout(); plt.show()

## 5. Train the first baseline
After inspection, change `DATA_REVIEWED` to `True`. Start with YOLOv8s at 640. Keep the seed and split unchanged between experiments. The settings below are starting values, not a guarantee of accuracy. Reduce `BATCH` to 4 or 2 if GPU memory runs out. More epochs alone will not repair bad data.

In [ ]:
from ultralytics import YOLO
DATA_REVIEWED = False  # Set True after completing the audit and visual/source review.
assert DATA_REVIEWED, "Complete the dataset review above"
MODEL_SOURCE = "yolov8s.pt"
IMGSZ = 640
BATCH = 8
RUN_NAME = "s640_india_v1"  # Use a new name for every fresh experiment.
assert not (PROJECT / RUN_NAME).exists(), "Choose a new RUN_NAME, or resume the interrupted run"
model = YOLO(MODEL_SOURCE)
model.train(
    data=str(DATA_YAML), epochs=100, patience=20,
    imgsz=IMGSZ, batch=BATCH, device=0, workers=2,
    optimizer="AdamW", lr0=0.001, lrf=0.01,
    weight_decay=0.0005, warmup_epochs=3, cos_lr=True,
    fliplr=0.0, flipud=0.0, degrees=5.0,
    translate=0.1, scale=0.3, hsv_h=0.005, hsv_s=0.3, hsv_v=0.3,
    mosaic=0.5, close_mosaic=10, mixup=0.0,
    seed=42, deterministic=True, amp=True, cache=False,
    project=str(PROJECT), name=RUN_NAME, save=True, save_period=5, plots=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights/best.pt"
print("Saved best:", BEST)

## 6. Validate and compare
This uses validation, not the test set. Record per-class results and inspect the confusion matrix.

Optional experiments, one at a time by rerunning cell 5 with new settings:
- `s960_india_v1`: same `yolov8s.pt`, `IMGSZ=960`, reduce batch if needed.
- `m640_india_v1`: `yolov8m.pt`, `IMGSZ=640`, reduce batch if needed. This isolates model size from resolution.
- `transfer_s640_v1`: upload your original `best.pt` to Colab, set `MODEL_SOURCE` to its path, keep 640. The changed class output is rebuilt for the dataset. Do not use `resume=True` to expand classes.

Evaluate the original four-class model only on compatible labels with an explicitly remapped ground truth. Its numeric class IDs cannot be directly scored against the 85-class YAML.


In [ ]:
trained = YOLO(str(BEST))
metrics = trained.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, device=0,
                      project=str(RUN_DIR), name="validation", plots=True)
print(metrics.results_dict)
print("Validation speed (ms/image):", metrics.speed)
rows = []
for j, cls_id in enumerate(metrics.box.ap_class_index):
    rows.append({"class_id": int(cls_id), "name": trained.names[int(cls_id)],
                 "precision": float(metrics.box.p[j]), "recall": float(metrics.box.r[j]),
                 "AP50": float(metrics.box.ap50[j]), "AP50_95": float(metrics.box.ap[j])})
pd.DataFrame(rows).to_csv(RUN_DIR / "per_class_validation.csv", index=False)
display(pd.DataFrame(rows))
(RUN_DIR / "validation_summary.json").write_text(json.dumps(
    {"metrics": metrics.results_dict, "speed": metrics.speed, "imgsz": IMGSZ,
     "ultralytics": ultralytics.__version__}, indent=2))

## 7. Final held-out evaluation (after selecting a model)
Leave this disabled while experimenting. Select the best run using validation metrics and speed, then evaluate once. Set `FINAL_DATA_YAML` to a separate labeled Indian road dataset with the **exact same ordered names** for stronger evidence of real-road generalization. This notebook checks names before evaluation.

The public dataset test split is useful, but does not by itself prove generalization to different cameras or locations. Unknown classes outside the trained vocabulary are not evaluated as recognized classes.


In [ ]:
RUN_FINAL_TEST = False
FINAL_BEST = BEST  # Change to the chosen run's weights/best.pt if necessary.
FINAL_IMGSZ = IMGSZ
FINAL_DATA_YAML = DATA_YAML  # Or an independent external evaluation YAML with test images.
if RUN_FINAL_TEST:
    final_cfg = yaml.safe_load(Path(FINAL_DATA_YAML).read_text())
    assert final_cfg.get("test"), "Provide a real held-out test split; do not substitute validation"
    final_names = final_cfg["names"]
    if isinstance(final_names, dict): final_names = [v for k,v in sorted(final_names.items(), key=lambda kv:int(kv[0]))]
    final_model = YOLO(str(FINAL_BEST))
    assert final_names == [final_model.names[i] for i in range(len(final_model.names))], "Class mapping mismatch"
    test_metrics = final_model.val(data=str(FINAL_DATA_YAML), split="test", imgsz=FINAL_IMGSZ,
                                  device=0, project=str(PROJECT), name="final_test", plots=True)
    print(test_metrics.results_dict)
else:
    print("Final test skipped until model selection is complete.")

## 8. Export the selected model for Flask
This downloads a model bundle. The trained file is named `best_india.pt`, preserving your old `best.pt`. The JSON is an audit record; the app reads labels from the model itself. Use the same Ultralytics version shown in the bundle on your laptop.

In [ ]:
import hashlib
EXPORT = PROJECT.parent / "export"
EXPORT.mkdir(exist_ok=True)
shutil.copy(FINAL_BEST, EXPORT / "best_india.pt")
selected = YOLO(str(FINAL_BEST))
metadata = {"names": selected.names, "imgsz": FINAL_IMGSZ,
            "source": str(FINAL_BEST), "ultralytics": ultralytics.__version__,
            "weights_sha256": hashlib.sha256((EXPORT / "best_india.pt").read_bytes()).hexdigest(),
            "dataset": "SDI Indian Traffic Sign", "license": "CC BY 4.0",
            "source_url": "https://universe.roboflow.com/sdi/indian-traffic-sign"}
(EXPORT / "model_metadata.json").write_text(json.dumps(metadata, indent=2))
(EXPORT / "requirements-model.txt").write_text(f"ultralytics=={ultralytics.__version__}\n")
archive_path = shutil.make_archive(str(PROJECT.parent / "roadsense_trained_model"), "zip", EXPORT)
files.download(archive_path)

## Recovery after a Colab disconnect
Mount Drive, reinstall dependencies, and re-extract/audit the same dataset to the same `/content` path. Then run the optional cell below. `last.pt` must be from an interrupted run, not a completed/stripped checkpoint. A completed run needs a new fine-tuning experiment, not resume. Free GPU allocation/session duration is not guaranteed.


In [ ]:
RESUME_INTERRUPTED = False
if RESUME_INTERRUPTED:
    checkpoint = PROJECT / "s640_india_v1/weights/last.pt"
    assert checkpoint.exists()
    resumed = YOLO(str(checkpoint))
    resumed.train(resume=True)
    RUN_DIR = Path(resumed.trainer.save_dir)
    BEST = RUN_DIR / "weights/best.pt"
    print("Resumed run saved:", BEST)